# SGLang 推理服务性能监控 — 简易实现方案

**定位：演示如何对 SGLang 推理服务进行健康检查、GPU 监控、请求性能统计及自动化监控循环，封装为可复用的监控类。**

> **默认启动命令：**
> ```bash
> python -m sglang.launch_server --model-path Qwen/Qwen2.5-0.5B-Instruct --host 127.0.0.1 --port 30000
> ```

*本 Notebook 为原创示例代码，可自由用于学习、教学及发布到 Gitee 等平台。*

---

## 硬件与环境要求

| 项目 | 最低要求 |
|------|----------|
| **GPU** | NVIDIA GPU（算力 ≥ 7.0） |
| **显存（VRAM）** | ≥ 6 GB（Qwen2.5-0.5B） |
| **驱动** | NVIDIA Driver ≥ 525 |
| **CUDA** | ≥ 12.1 |
| **Python** | 3.9 – 3.12 |
| **操作系统** | Linux x86_64 最佳；Windows 建议使用 WSL2 |

---

## 依赖安装

```bash
# 安装 SGLang 全部组件及 OpenAI SDK
pip install "sglang[all]" "openai>=1.0.0" "requests>=2.28.0"
```

---

## 使用指南

1. 按顺序执行每个代码单元格
2. 先启动 SGLang 服务，等待模型加载完成
3. 依次体验健康检查、GPU 监控、性能统计等功能
4. 最后运行「封装监控类」单元格获取可复用的监控工具
5. 执行完毕后务必运行「清理资源」单元格释放 GPU 显存

---

In [ ]:
# === 环境检查 ===
import subprocess  # 导入子进程模块，用于执行系统命令
import sys  # 导入系统模块，获取 Python 版本信息

print("=" * 50)  # 打印分隔线
print("环境检查")  # 打印标题
print("=" * 50)  # 打印分隔线

# 检查 Python 版本
print(f"Python 版本: {sys.version}")  # 输出当前 Python 版本

# 检查 GPU 是否可用
try:
    result = subprocess.run(  # 执行 nvidia-smi 命令查询 GPU 信息
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],  # 查询 GPU 名称和显存
        capture_output=True, text=True, timeout=10  # 捕获输出，设置超时
    )
    if result.returncode == 0:  # 如果命令执行成功
        print(f"GPU 信息: {result.stdout.strip()}")  # 输出 GPU 信息
    else:
        print("警告: nvidia-smi 执行失败")  # 输出警告
except FileNotFoundError:  # 如果找不到 nvidia-smi
    print("警告: 未找到 nvidia-smi，请确认已安装 NVIDIA 驱动")  # 输出警告

# 检查 sglang 是否已安装
try:
    import sglang  # 尝试导入 sglang
    print(f"SGLang 版本: {sglang.__version__}")  # 输出 sglang 版本
except ImportError:  # 如果未安装
    print("SGLang 未安装，请运行下方安装单元格")  # 提示安装

# 检查 openai 包
try:
    import openai  # 尝试导入 openai
    print(f"OpenAI SDK 版本: {openai.__version__}")  # 输出 openai 版本
except ImportError:  # 如果未安装
    print("OpenAI SDK 未安装")  # 提示安装

print("=" * 50)  # 打印分隔线

In [ ]:
# === 可选：安装依赖（已安装可跳过） ===
# !pip install "sglang[all]" "openai>=1.0.0" "requests>=2.28.0"  # 取消注释以安装依赖

## 为什么需要监控？

在生产环境中部署 LLM 推理服务时，监控至关重要：

1. **保障 SLA（服务等级协议）**：实时感知延迟、吞吐量是否达标
2. **及时发现异常**：服务宕机、GPU OOM、延迟突增等问题需要第一时间告警
3. **容量规划**：基于历史数据判断何时需要扩容
4. **调优依据**：性能指标帮助定位瓶颈并验证优化效果

本 Notebook 实现一套**轻量级监控方案**，包含：
- 健康检查（Health Check）
- GPU 资源监控
- 请求级别性能统计（延迟、吞吐）
- 简易监控循环
- 监控报告生成
- 可复用的监控类封装

In [ ]:
# === 启动 SGLang 服务 ===
import subprocess  # 导入子进程模块
import time  # 导入时间模块
import requests  # 导入 HTTP 请求模块

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"  # 定义使用的模型 ID
HOST = "127.0.0.1"  # 定义服务监听地址
PORT = 30000  # 定义服务监听端口

# 构建启动命令
launch_cmd = [  # 启动命令列表
    "python", "-m", "sglang.launch_server",  # 使用 sglang 启动服务
    "--model-path", MODEL_ID,  # 指定模型路径
    "--host", HOST,  # 指定监听地址
    "--port", str(PORT),  # 指定监听端口
]

print(f"启动命令: {' '.join(launch_cmd)}")  # 打印启动命令
print("正在启动 SGLang 服务...")  # 提示正在启动

# 以子进程方式启动服务
process = subprocess.Popen(  # 启动后台进程
    launch_cmd,  # 传入命令列表
    stdout=subprocess.PIPE,  # 捕获标准输出
    stderr=subprocess.STDOUT,  # 将标准错误合并到标准输出
    text=True  # 以文本模式读取输出
)

# 等待服务就绪
BASE_URL = f"http://{HOST}:{PORT}"  # 构建服务基础 URL
max_wait = 180  # 最大等待时间（秒）
start_time = time.time()  # 记录开始时间

while time.time() - start_time < max_wait:  # 在最大等待时间内循环
    try:
        resp = requests.get(f"{BASE_URL}/v1/models", timeout=5)  # 尝试请求模型列表
        if resp.status_code == 200:  # 如果返回 200
            elapsed = time.time() - start_time  # 计算等待耗时
            print(f"\n服务已就绪! (等待了 {elapsed:.1f} 秒)")  # 输出就绪信息
            print(f"可用模型: {[m['id'] for m in resp.json()['data']]}")  # 输出模型列表
            break  # 跳出循环
    except requests.ConnectionError:  # 如果连接失败
        pass  # 忽略，继续等待
    time.sleep(3)  # 每 3 秒重试一次
    print(".", end="", flush=True)  # 打印进度点
else:
    print(f"\n警告: 等待 {max_wait} 秒后服务仍未就绪")  # 超时警告

## 健康检查

健康检查是最基础的监控手段，用于判断服务是否存活、能否正常响应请求。

In [ ]:
# === 健康检查 ===
import requests  # 导入 HTTP 请求模块
import time  # 导入时间模块

def check_health(base_url="http://127.0.0.1:30000", timeout=10):  # 定义健康检查函数
    """检查 SGLang 服务健康状态"""
    result = {  # 初始化结果字典
        "status": "unknown",  # 默认状态为未知
        "response_time_ms": None,  # 响应时间初始为空
        "models": [],  # 可用模型列表初始为空
        "error": None  # 错误信息初始为空
    }
    
    start = time.time()  # 记录请求开始时间
    try:
        resp = requests.get(f"{base_url}/v1/models", timeout=timeout)  # 请求模型列表接口
        elapsed_ms = (time.time() - start) * 1000  # 计算响应时间（毫秒）
        result["response_time_ms"] = round(elapsed_ms, 2)  # 记录响应时间
        
        if resp.status_code == 200:  # 如果响应状态码为 200
            result["status"] = "healthy"  # 标记为健康
            data = resp.json()  # 解析 JSON 响应
            result["models"] = [m["id"] for m in data.get("data", [])]  # 提取模型 ID 列表
        else:
            result["status"] = "unhealthy"  # 标记为不健康
            result["error"] = f"HTTP {resp.status_code}"  # 记录 HTTP 错误码
    except requests.ConnectionError:  # 如果连接失败
        result["status"] = "unreachable"  # 标记为不可达
        result["error"] = "连接被拒绝"  # 记录错误信息
    except requests.Timeout:  # 如果请求超时
        result["status"] = "timeout"  # 标记为超时
        result["error"] = f"请求超时（>{timeout}秒）"  # 记录超时信息
    except Exception as e:  # 捕获其他异常
        result["status"] = "error"  # 标记为错误
        result["error"] = str(e)  # 记录异常信息
    
    return result  # 返回检查结果

# 执行健康检查
print("=" * 50)  # 打印分隔线
print("SGLang 服务健康检查")  # 打印标题
print("=" * 50)  # 打印分隔线

health = check_health()  # 调用健康检查函数
print(f"服务状态: {health['status']}")  # 输出服务状态
print(f"响应时间: {health['response_time_ms']} ms")  # 输出响应时间
print(f"可用模型: {health['models']}")  # 输出可用模型
if health["error"]:  # 如果有错误信息
    print(f"错误信息: {health['error']}")  # 输出错误

## GPU 资源监控

通过 `nvidia-smi` 获取 GPU 实时状态，包括利用率、显存使用和温度。

In [ ]:
# === GPU 资源监控 ===
import subprocess  # 导入子进程模块

def get_gpu_stats():  # 定义获取 GPU 统计信息的函数
    """通过 nvidia-smi 获取 GPU 资源使用情况"""
    gpu_list = []  # 初始化 GPU 信息列表
    try:
        result = subprocess.run(  # 执行 nvidia-smi 查询命令
            ["nvidia-smi",  # 调用 nvidia-smi
             "--query-gpu=index,name,utilization.gpu,memory.used,memory.total,temperature.gpu",  # 查询 GPU 各项指标
             "--format=csv,noheader,nounits"],  # 以 CSV 格式输出，不带表头和单位
            capture_output=True, text=True, timeout=10  # 捕获输出，设置超时
        )
        if result.returncode == 0:  # 如果命令执行成功
            for line in result.stdout.strip().split("\n"):  # 逐行解析输出
                parts = [p.strip() for p in line.split(",")]  # 按逗号分割并去除空格
                if len(parts) >= 6:  # 确保字段数量足够
                    gpu_info = {  # 构建 GPU 信息字典
                        "gpu_id": int(parts[0]),  # GPU 编号
                        "name": parts[1],  # GPU 型号名称
                        "utilization_pct": int(parts[2]),  # GPU 利用率百分比
                        "memory_used_mb": int(parts[3]),  # 已使用显存（MB）
                        "memory_total_mb": int(parts[4]),  # 总显存（MB）
                        "temperature_c": int(parts[5]),  # GPU 温度（摄氏度）
                    }
                    gpu_info["memory_used_pct"] = round(  # 计算显存使用百分比
                        gpu_info["memory_used_mb"] / gpu_info["memory_total_mb"] * 100, 1  # 百分比保留一位小数
                    )
                    gpu_list.append(gpu_info)  # 添加到列表
    except FileNotFoundError:  # 如果找不到 nvidia-smi
        print("错误: 未找到 nvidia-smi")  # 输出错误信息
    except Exception as e:  # 捕获其他异常
        print(f"获取 GPU 信息失败: {e}")  # 输出错误
    
    return gpu_list  # 返回 GPU 信息列表

# 执行 GPU 监控
print("=" * 60)  # 打印分隔线
print("GPU 资源使用情况")  # 打印标题
print("=" * 60)  # 打印分隔线

gpu_stats = get_gpu_stats()  # 调用获取 GPU 信息的函数
for gpu in gpu_stats:  # 遍历每块 GPU
    print(f"\nGPU {gpu['gpu_id']}: {gpu['name']}")  # 输出 GPU 编号和名称
    print(f"  利用率: {gpu['utilization_pct']}%")  # 输出 GPU 利用率
    print(f"  显存: {gpu['memory_used_mb']}MB / {gpu['memory_total_mb']}MB ({gpu['memory_used_pct']}%)")  # 输出显存使用
    print(f"  温度: {gpu['temperature_c']}°C")  # 输出温度

if not gpu_stats:  # 如果没有获取到 GPU 信息
    print("未检测到 GPU 信息")  # 输出提示

## 请求级别性能统计

发送多个请求，收集每个请求的详细性能指标：
- **首 token 延迟（TTFT）**：从发送请求到收到第一个 token 的时间
- **总延迟**：完成整个请求的时间
- **生成 token 数**：模型输出的 token 数量
- **吞吐量（TPS）**：每秒生成的 token 数

最终计算 avg、p50、p90、p99 等分位数。

In [ ]:
# === 请求级别性能统计 ===
import time  # 导入时间模块
import statistics  # 导入统计模块
from openai import OpenAI  # 从 openai 包导入客户端

def benchmark_request(client, model_id, prompt="请用一句话介绍人工智能", max_tokens=100):  # 定义单次请求基准测试函数
    """发送单次请求并收集性能指标"""
    metrics = {}  # 初始化指标字典
    
    # 使用流式请求以测量首 token 延迟
    start_time = time.time()  # 记录请求开始时间
    first_token_time = None  # 初始化首 token 时间
    token_count = 0  # 初始化 token 计数器
    full_text = ""  # 初始化完整文本
    
    try:
        stream = client.chat.completions.create(  # 发送流式聊天请求
            model=model_id,  # 指定模型
            messages=[{"role": "user", "content": prompt}],  # 设置用户消息
            max_tokens=max_tokens,  # 限制最大生成 token 数
            stream=True  # 启用流式输出
        )
        
        for chunk in stream:  # 遍历流式响应的每个块
            if chunk.choices[0].delta.content:  # 如果当前块包含内容
                if first_token_time is None:  # 如果是第一个 token
                    first_token_time = time.time()  # 记录首 token 时间
                token_count += 1  # 增加 token 计数
                full_text += chunk.choices[0].delta.content  # 拼接文本内容
        
        end_time = time.time()  # 记录请求结束时间
        
        metrics["ttft_ms"] = round((first_token_time - start_time) * 1000, 2) if first_token_time else None  # 首 token 延迟（毫秒）
        metrics["total_latency_ms"] = round((end_time - start_time) * 1000, 2)  # 总延迟（毫秒）
        metrics["tokens_generated"] = token_count  # 生成的 token 数
        total_seconds = end_time - start_time  # 总耗时（秒）
        metrics["tokens_per_second"] = round(token_count / total_seconds, 2) if total_seconds > 0 else 0  # 每秒 token 数
        metrics["success"] = True  # 标记为成功
    except Exception as e:  # 捕获异常
        metrics["success"] = False  # 标记为失败
        metrics["error"] = str(e)  # 记录错误信息
    
    return metrics  # 返回性能指标

def calculate_percentile(data, percentile):  # 定义计算分位数的函数
    """计算给定数据的指定分位数"""
    sorted_data = sorted(data)  # 对数据排序
    index = int(len(sorted_data) * percentile / 100)  # 计算分位数索引
    index = min(index, len(sorted_data) - 1)  # 防止索引越界
    return sorted_data[index]  # 返回分位数值

# 执行多次请求并统计性能
print("=" * 60)  # 打印分隔线
print("请求级别性能统计")  # 打印标题
print("=" * 60)  # 打印分隔线

client = OpenAI(  # 创建 OpenAI 客户端
    base_url="http://127.0.0.1:30000/v1",  # 设置 API 地址
    api_key="EMPTY"  # SGLang 不需要真实 Key
)

N = 5  # 定义测试请求次数
all_metrics = []  # 初始化指标收集列表

for i in range(N):  # 循环发送 N 次请求
    print(f"发送请求 {i+1}/{N}...", end=" ")  # 打印进度
    m = benchmark_request(client, "Qwen/Qwen2.5-0.5B-Instruct")  # 执行基准测试
    all_metrics.append(m)  # 收集指标
    if m["success"]:  # 如果请求成功
        print(f"延迟={m['total_latency_ms']}ms, TTFT={m['ttft_ms']}ms, TPS={m['tokens_per_second']}")  # 输出指标
    else:
        print(f"失败: {m.get('error', '未知错误')}")  # 输出错误

# 计算汇总统计
successful = [m for m in all_metrics if m["success"]]  # 过滤成功的请求
if successful:  # 如果有成功的请求
    latencies = [m["total_latency_ms"] for m in successful]  # 提取延迟列表
    ttfts = [m["ttft_ms"] for m in successful if m["ttft_ms"] is not None]  # 提取 TTFT 列表
    tps_list = [m["tokens_per_second"] for m in successful]  # 提取 TPS 列表
    
    print(f"\n{'=' * 60}")  # 打印分隔线
    print(f"汇总统计（{len(successful)}/{N} 次请求成功）")  # 打印统计标题
    print(f"{'=' * 60}")  # 打印分隔线
    print(f"总延迟 - 平均: {statistics.mean(latencies):.1f}ms, "
          f"P50: {calculate_percentile(latencies, 50):.1f}ms, "
          f"P90: {calculate_percentile(latencies, 90):.1f}ms, "
          f"P99: {calculate_percentile(latencies, 99):.1f}ms")  # 输出延迟统计
    if ttfts:  # 如果有 TTFT 数据
        print(f"首token - 平均: {statistics.mean(ttfts):.1f}ms, "
              f"P50: {calculate_percentile(ttfts, 50):.1f}ms, "
              f"P90: {calculate_percentile(ttfts, 90):.1f}ms")  # 输出 TTFT 统计
    print(f"吞吐量 - 平均: {statistics.mean(tps_list):.1f} tokens/s")  # 输出吞吐量

## 简易监控循环

将健康检查、GPU 监控和请求测试整合到一个循环中，每隔一定时间间隔采集一次数据。

> 以下示例运行 15 秒，每 3 秒采集一次。实际生产中可调整为更长的运行时间和间隔。

In [ ]:
# === 简易监控循环 ===
import time  # 导入时间模块
import datetime  # 导入日期时间模块

MONITOR_DURATION = 15  # 监控总时长（秒）
MONITOR_INTERVAL = 3  # 采样间隔（秒）

monitoring_data = []  # 初始化监控数据收集列表

print("=" * 60)  # 打印分隔线
print(f"开始监控循环（持续 {MONITOR_DURATION} 秒，间隔 {MONITOR_INTERVAL} 秒）")  # 打印监控参数
print("=" * 60)  # 打印分隔线

loop_start = time.time()  # 记录循环开始时间

while time.time() - loop_start < MONITOR_DURATION:  # 在监控时间内循环
    tick_start = time.time()  # 记录当前采样开始时间
    timestamp = datetime.datetime.now().strftime("%H:%M:%S")  # 获取当前时间戳
    
    # 采集各项指标
    health = check_health()  # 执行健康检查
    gpu_stats = get_gpu_stats()  # 获取 GPU 统计
    req_metrics = benchmark_request(client, "Qwen/Qwen2.5-0.5B-Instruct", max_tokens=30)  # 发送测试请求
    
    # 整合单次采样数据
    sample = {  # 构建采样数据字典
        "timestamp": timestamp,  # 时间戳
        "health_status": health["status"],  # 服务健康状态
        "health_response_ms": health["response_time_ms"],  # 健康检查响应时间
        "gpu_utilization": gpu_stats[0]["utilization_pct"] if gpu_stats else None,  # GPU 利用率
        "gpu_memory_pct": gpu_stats[0]["memory_used_pct"] if gpu_stats else None,  # GPU 显存使用率
        "gpu_temperature": gpu_stats[0]["temperature_c"] if gpu_stats else None,  # GPU 温度
        "request_latency_ms": req_metrics.get("total_latency_ms"),  # 请求延迟
        "request_ttft_ms": req_metrics.get("ttft_ms"),  # 首 token 延迟
        "request_tps": req_metrics.get("tokens_per_second"),  # 吞吐量
        "request_success": req_metrics.get("success", False),  # 请求是否成功
    }
    monitoring_data.append(sample)  # 添加到监控数据列表
    
    # 实时打印当前采样结果
    status_icon = "✓" if health["status"] == "healthy" else "✗"  # 根据状态选择图标
    gpu_info = f"GPU {sample['gpu_utilization']}%" if sample["gpu_utilization"] is not None else "N/A"  # GPU 信息
    latency_info = f"{sample['request_latency_ms']}ms" if sample["request_latency_ms"] else "失败"  # 延迟信息
    print(f"[{timestamp}] {status_icon} 服务={health['status']} | {gpu_info} | 延迟={latency_info}")  # 输出采样摘要
    
    # 等待到下次采样
    elapsed = time.time() - tick_start  # 计算本次采样耗时
    sleep_time = max(0, MONITOR_INTERVAL - elapsed)  # 计算需要等待的时间
    time.sleep(sleep_time)  # 等待到下次采样

print(f"\n监控结束，共采集 {len(monitoring_data)} 个数据点")  # 输出监控结束信息

## 监控报告生成

对监控循环收集的数据进行聚合分析，生成一份简易的文字报告。

In [ ]:
# === 监控报告生成 ===
import statistics  # 导入统计模块

def generate_report(data):  # 定义报告生成函数
    """根据监控数据生成汇总报告"""
    if not data:  # 如果没有数据
        print("无监控数据可用")  # 输出提示
        return  # 直接返回
    
    total_samples = len(data)  # 总采样次数
    healthy_count = sum(1 for d in data if d["health_status"] == "healthy")  # 统计健康次数
    success_count = sum(1 for d in data if d["request_success"])  # 统计请求成功次数
    
    # 提取各指标列表
    latencies = [d["request_latency_ms"] for d in data if d["request_latency_ms"] is not None]  # 延迟列表
    ttfts = [d["request_ttft_ms"] for d in data if d["request_ttft_ms"] is not None]  # TTFT 列表
    gpu_utils = [d["gpu_utilization"] for d in data if d["gpu_utilization"] is not None]  # GPU 利用率列表
    gpu_mems = [d["gpu_memory_pct"] for d in data if d["gpu_memory_pct"] is not None]  # GPU 显存使用率列表
    gpu_temps = [d["gpu_temperature"] for d in data if d["gpu_temperature"] is not None]  # GPU 温度列表
    tps_values = [d["request_tps"] for d in data if d["request_tps"] is not None]  # 吞吐量列表
    
    # 打印报告
    print("=" * 60)  # 打印分隔线
    print("           SGLang 监控报告")  # 打印报告标题
    print("=" * 60)  # 打印分隔线
    
    # 基本信息
    print(f"\n【基本信息】")  # 打印章节标题
    print(f"  监控时段: {data[0]['timestamp']} ~ {data[-1]['timestamp']}")  # 输出时间范围
    print(f"  采样总数: {total_samples}")  # 输出采样总数
    print(f"  服务可用率: {healthy_count}/{total_samples} ({healthy_count/total_samples*100:.1f}%)")  # 输出可用率
    print(f"  请求成功率: {success_count}/{total_samples} ({success_count/total_samples*100:.1f}%)")  # 输出成功率
    
    # 延迟统计
    if latencies:  # 如果有延迟数据
        print(f"\n【延迟统计】")  # 打印章节标题
        print(f"  平均延迟: {statistics.mean(latencies):.1f} ms")  # 输出平均延迟
        print(f"  最小延迟: {min(latencies):.1f} ms")  # 输出最小延迟
        print(f"  最大延迟: {max(latencies):.1f} ms")  # 输出最大延迟
    
    if ttfts:  # 如果有 TTFT 数据
        print(f"  平均 TTFT: {statistics.mean(ttfts):.1f} ms")  # 输出平均首 token 延迟
    
    if tps_values:  # 如果有吞吐量数据
        print(f"  平均吞吐: {statistics.mean(tps_values):.1f} tokens/s")  # 输出平均吞吐量
    
    # GPU 统计
    if gpu_utils:  # 如果有 GPU 利用率数据
        print(f"\n【GPU 资源】")  # 打印章节标题
        print(f"  GPU 利用率: 平均 {statistics.mean(gpu_utils):.1f}%, 最高 {max(gpu_utils)}%")  # 输出利用率
    if gpu_mems:  # 如果有显存数据
        print(f"  显存使用: 平均 {statistics.mean(gpu_mems):.1f}%, 最高 {max(gpu_mems):.1f}%")  # 输出显存
    if gpu_temps:  # 如果有温度数据
        print(f"  GPU 温度: 平均 {statistics.mean(gpu_temps):.1f}°C, 最高 {max(gpu_temps)}°C")  # 输出温度
    
    # 简易 ASCII 可视化：延迟趋势
    if latencies:  # 如果有延迟数据
        print(f"\n【延迟趋势】")  # 打印章节标题
        max_lat = max(latencies)  # 获取最大延迟
        bar_width = 30  # 设置柱状图宽度
        for i, lat in enumerate(latencies):  # 遍历每个延迟值
            bar_len = int(lat / max_lat * bar_width) if max_lat > 0 else 0  # 计算柱状图长度
            bar = "█" * bar_len  # 生成柱状图
            print(f"  #{i+1:2d} |{bar:<{bar_width}}| {lat:.0f}ms")  # 输出一行柱状图
    
    print(f"\n{'=' * 60}")  # 打印分隔线

# 生成报告
generate_report(monitoring_data)  # 调用报告生成函数

## 封装为可复用的监控类

将上述功能整合到 `SGLangMonitor` 类中，方便在不同项目中复用。

In [ ]:
# === 封装为可复用的监控类 ===
import time  # 导入时间模块
import datetime  # 导入日期时间模块
import statistics  # 导入统计模块
import subprocess  # 导入子进程模块
import requests  # 导入 HTTP 请求模块
from openai import OpenAI  # 从 openai 包导入客户端

class SGLangMonitor:  # 定义 SGLang 监控类
    """SGLang 推理服务监控工具"""
    
    def __init__(self, base_url="http://127.0.0.1:30000", model_id="Qwen/Qwen2.5-0.5B-Instruct"):  # 初始化方法
        self.base_url = base_url  # 保存服务地址
        self.model_id = model_id  # 保存模型 ID
        self.client = OpenAI(base_url=f"{base_url}/v1", api_key="EMPTY")  # 创建 OpenAI 客户端
        self.history = []  # 初始化历史数据列表
    
    def check_health(self, timeout=10):  # 定义健康检查方法
        """检查服务健康状态"""
        result = {"status": "unknown", "response_time_ms": None, "error": None}  # 初始化结果
        start = time.time()  # 记录开始时间
        try:
            resp = requests.get(f"{self.base_url}/v1/models", timeout=timeout)  # 请求模型列表
            result["response_time_ms"] = round((time.time() - start) * 1000, 2)  # 计算响应时间
            result["status"] = "healthy" if resp.status_code == 200 else "unhealthy"  # 判断健康状态
        except requests.ConnectionError:  # 如果连接失败
            result["status"] = "unreachable"  # 标记为不可达
            result["error"] = "连接被拒绝"  # 记录错误
        except requests.Timeout:  # 如果超时
            result["status"] = "timeout"  # 标记为超时
            result["error"] = "请求超时"  # 记录错误
        return result  # 返回结果
    
    def get_gpu_stats(self):  # 定义获取 GPU 信息的方法
        """获取 GPU 资源使用情况"""
        gpu_list = []  # 初始化列表
        try:
            result = subprocess.run(  # 执行 nvidia-smi 命令
                ["nvidia-smi", "--query-gpu=index,name,utilization.gpu,memory.used,memory.total,temperature.gpu",
                 "--format=csv,noheader,nounits"],  # 查询参数
                capture_output=True, text=True, timeout=10  # 捕获输出
            )
            if result.returncode == 0:  # 如果成功
                for line in result.stdout.strip().split("\n"):  # 逐行解析
                    parts = [p.strip() for p in line.split(",")]  # 分割字段
                    if len(parts) >= 6:  # 确保字段完整
                        gpu_list.append({  # 添加 GPU 信息
                            "gpu_id": int(parts[0]),  # GPU 编号
                            "utilization_pct": int(parts[2]),  # 利用率
                            "memory_used_mb": int(parts[3]),  # 已用显存
                            "memory_total_mb": int(parts[4]),  # 总显存
                            "temperature_c": int(parts[5]),  # 温度
                        })
        except Exception:  # 捕获所有异常
            pass  # 静默处理
        return gpu_list  # 返回 GPU 列表
    
    def benchmark_request(self, prompt="你好", max_tokens=50):  # 定义请求基准测试方法
        """发送测试请求并返回性能指标"""
        metrics = {}  # 初始化指标字典
        start = time.time()  # 记录开始时间
        first_token_time = None  # 初始化首 token 时间
        token_count = 0  # 初始化计数器
        try:
            stream = self.client.chat.completions.create(  # 发送流式请求
                model=self.model_id,  # 指定模型
                messages=[{"role": "user", "content": prompt}],  # 设置消息
                max_tokens=max_tokens, stream=True  # 启用流式
            )
            for chunk in stream:  # 遍历流式块
                if chunk.choices[0].delta.content:  # 如果有内容
                    if first_token_time is None:  # 如果是首个 token
                        first_token_time = time.time()  # 记录时间
                    token_count += 1  # 计数加一
            end = time.time()  # 记录结束时间
            metrics["ttft_ms"] = round((first_token_time - start) * 1000, 2) if first_token_time else None  # 首 token 延迟
            metrics["total_latency_ms"] = round((end - start) * 1000, 2)  # 总延迟
            metrics["tokens_per_second"] = round(token_count / (end - start), 2) if end > start else 0  # 吞吐量
            metrics["success"] = True  # 标记成功
        except Exception as e:  # 捕获异常
            metrics["success"] = False  # 标记失败
            metrics["error"] = str(e)  # 记录错误
        return metrics  # 返回指标
    
    def collect_sample(self):  # 定义采样方法
        """采集一次完整的监控样本"""
        health = self.check_health()  # 执行健康检查
        gpu = self.get_gpu_stats()  # 获取 GPU 信息
        req = self.benchmark_request()  # 执行测试请求
        sample = {  # 构建采样字典
            "timestamp": datetime.datetime.now().isoformat(),  # 时间戳
            "health": health,  # 健康检查结果
            "gpu": gpu,  # GPU 信息
            "request": req,  # 请求性能
        }
        self.history.append(sample)  # 添加到历史记录
        return sample  # 返回采样
    
    def generate_report(self):  # 定义报告生成方法
        """根据历史数据生成监控报告"""
        if not self.history:  # 如果没有历史数据
            return "无数据"  # 返回提示
        total = len(self.history)  # 总样本数
        healthy = sum(1 for s in self.history if s["health"]["status"] == "healthy")  # 健康次数
        latencies = [s["request"]["total_latency_ms"] for s in self.history  # 延迟列表
                     if s["request"].get("success") and s["request"].get("total_latency_ms")]  # 过滤成功请求
        report = f"采样数={total}, 可用率={healthy/total*100:.1f}%"  # 基本信息
        if latencies:  # 如果有延迟数据
            report += f", 平均延迟={statistics.mean(latencies):.1f}ms"  # 追加平均延迟
        return report  # 返回报告

# 演示使用监控类
print("=" * 60)  # 打印分隔线
print("SGLangMonitor 类演示")  # 打印标题
print("=" * 60)  # 打印分隔线

monitor = SGLangMonitor()  # 创建监控实例

# 采集 3 个样本
for i in range(3):  # 循环 3 次
    sample = monitor.collect_sample()  # 采集一次样本
    print(f"样本 {i+1}: 状态={sample['health']['status']}, "
          f"延迟={sample['request'].get('total_latency_ms', 'N/A')}ms")  # 输出采样结果
    time.sleep(1)  # 间隔 1 秒

# 生成报告
print(f"\n监控报告: {monitor.generate_report()}")  # 输出报告

In [ ]:
# === 清理资源 ===
import subprocess  # 导入子进程模块

print("正在清理 SGLang 服务进程...")  # 打印清理提示

# 终止本 Notebook 启动的进程
try:
    process.terminate()  # 发送终止信号给服务进程
    process.wait(timeout=10)  # 等待进程结束，超时 10 秒
    print(f"进程 {process.pid} 已终止")  # 输出终止确认
except NameError:  # 如果 process 变量未定义
    print("未找到活跃的服务进程变量")  # 提示未找到进程
except Exception as e:  # 捕获其他异常
    print(f"终止进程时出错: {e}")  # 输出错误信息

# 兜底：杀掉所有 sglang 相关进程
result = subprocess.run(  # 执行 pkill 命令
    ["pkill", "-f", "sglang.launch_server"],  # 按名称匹配并终止 sglang 进程
    capture_output=True, text=True  # 捕获输出
)
print("已尝试清理所有 SGLang 相关进程")  # 输出清理完成提示
print("GPU 显存将在几秒后释放")  # 提示显存释放

## 常见问题排查

### 问题 1：nvidia-smi 命令未找到

**现象：** 执行 GPU 监控时报 `FileNotFoundError: [Errno 2] No such file or directory: 'nvidia-smi'`。

**可能原因：**
- NVIDIA 驱动未安装
- nvidia-smi 不在系统 PATH 中
- 在容器中运行但未传入 GPU 设备

**解决：**
```bash
# 确认驱动已安装
ls /usr/bin/nvidia-smi || ls /usr/local/cuda/bin/nvidia-smi

# 如果在非标准路径，添加到 PATH
export PATH=$PATH:/usr/local/cuda/bin

# 如果在 Docker 容器中，确保使用 --gpus all
docker run --gpus all ...
```

### 问题 2：健康检查时常失败（间歇性超时）

**现象：** 监控循环中健康检查偶尔返回 `timeout` 状态。

**可能原因：**
- 服务正在处理大量并发请求，响应变慢
- 默认超时时间过短
- 网络抖动（尤其是远程部署场景）

**解决：** 增大健康检查的超时时间；减少测试请求的并发数；在监控逻辑中加入重试机制。

```python
# 增大超时时间
health = check_health(timeout=30)

# 或在监控类中调整
monitor = SGLangMonitor()
monitor.check_health(timeout=30)
```

---

### 结语

本 Notebook 实现了一套轻量级的 SGLang 推理服务监控方案，涵盖健康检查、GPU 监控、请求性能统计、自动化监控循环和报告生成。`SGLangMonitor` 类可直接复用到生产项目中，也可作为 Prometheus/Grafana 等专业监控方案的补充。